In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches, load_elo_baseline
from src.elo import prepare_matches, run_elo_updates, update_elo
from src.simulation import predict_match, Matchprediction
from src.features import update_team_history, update_h2h

In [2]:
draw_threshold = joblib.load('draw_threshold.joblib')
wc_model = joblib.load('world_cup_model.joblib')
history_dict = joblib.load('final_history.joblib')
h2h_dict = joblib.load('final_h2h.joblib')
country_elo = joblib.load('final_elo.joblib')

group_stage_matches = load_matches()
group_stage_matches = prepare_matches(group_stage_matches, "2026-06-10", "2026-06-28")

In [20]:
#Group stage simulation

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Congo DR", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}
wc_teams = [
    "Algeria", "Argentina", "Australia", "Austria", "Belgium", 
    "Bosnia and Herzegovina", "Brazil", "Cape Verde", "Colombia", 
    "Congo DR", "Croatia", "Curaçao", "Czech Republic", "Ecuador", "Egypt", 
    "England", "France", "Germany", "Ghana", "Haiti", "Iran", "Iraq", 
    "Japan", "Jordan", "Saudi Arabia", "South Africa", "South Korea", 
    "Ivory Coast", "Mexico", "Morocco", "Netherlands", "New Zealand", 
    "Norway", "Panama", "Paraguay", "Qatar", "Scotland", "Senegal", 
    "Spain", "Sweden", "Switzerland", "Tunisia", "Turkey", 
    "United States", "Uruguay", "Uzbekistan", "Canada"
]

rows=[]
for group, teams in wc_groups.items():
    for team in teams:
        rows.append({"Group": group,
                     "Team": team,
                     "Points": 0})
group_stage_result = pd.DataFrame(rows) #Gives us the starting group stage
for match in group_stage_matches.itertuples(index = False):
    x = predict_match(match.home_team, match.away_team, wc_model, draw_threshold, history_dict, h2h_dict, country_elo)
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'Points'] += x.home_points
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'Points'] += x.away_points
    
    if x.outcome == "Home Win": S = 1
    elif x.outcome == "Away Win": S = 0
    else: S = 0.5
    
    new_away, new_home = update_elo(S, match.neutral, match.K_factor, 0, 0, 
                                    country_elo[match.away_team], 
                                    country_elo[match.home_team])
    
    country_elo[match.home_team] = new_home
    country_elo[match.away_team] = new_away
    
    update_team_history(match, history_dict)
    update_h2h(match, h2h_dict)

group_stage_result = group_stage_result.sort_values(['Group', 'Points'], ascending=[True, False]).reset_index(drop=True)    
print(group_stage_result) #Predicted group stage results

   Group                    Team  Points
0      A                  Mexico       9
1      A             South Korea       4
2      A            South Africa       2
3      A          Czech Republic       1
4      B                  Canada       9
5      B             Switzerland       6
6      B  Bosnia and Herzegovina       3
7      B                   Qatar       0
8      C                  Brazil       7
9      C                Scotland       5
10     C                 Morocco       2
11     C                   Haiti       1
12     D           United States       7
13     D                  Turkey       5
14     D                Paraguay       2
15     D               Australia       1
16     E                 Germany       7
17     E                 Ecuador       5
18     E             Ivory Coast       4
19     E                 Curaçao       0
20     F             Netherlands       7
21     F                   Japan       5
22     F                  Sweden       4
23     F        